# 03 — Feature Store Construction
**Demand Signal Feature Store — Capstone Project**

This notebook builds the unified feature store:
1. **ERP features** — aggregated monthly supplier-level metrics from shipment data
2. **LLM features** — aggregated monthly supplier-level signals from extracted notes
3. **Combined store** — merged with lineage tracking and freshness SLA checks

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from src.feature_store import (
    build_erp_features,
    build_llm_features,
    build_combined_feature_store,
    check_freshness_sla,
    get_lineage_report,
)

## 1. Load Raw Data

In [ ]:
erp_df = pd.read_csv("../data/erp_shipments.csv", parse_dates=["order_date", "expected_delivery_date", "actual_delivery_date"])
notes_df = pd.read_csv("../data/supplier_notes.csv", parse_dates=["note_date"])
extracted_df = pd.read_csv("../data/extracted_signals.csv")

print(f"ERP records: {len(erp_df)}")
print(f"Supplier notes: {len(notes_df)}")
print(f"Extracted signals: {len(extracted_df)}")

## 2. Build ERP Feature Family

In [ ]:
erp_features = build_erp_features(erp_df)
print(f"ERP features: {erp_features.shape}")
print(f"Columns: {list(erp_features.columns)}")
erp_features.head()

## 3. Build LLM Feature Family

In [ ]:
llm_features = build_llm_features(notes_df, extracted_df)
print(f"LLM features: {llm_features.shape}")
print(f"Columns: {list(llm_features.columns)}")
llm_features.head()

## 4. Build Combined Feature Store

In [ ]:
feature_store = build_combined_feature_store(erp_features, llm_features)
print(f"Combined feature store: {feature_store.shape}")
print(f"\nAll columns:")
for col in feature_store.columns:
    print(f"  {col}: {feature_store[col].dtype}")
feature_store.head()

## 5. Freshness SLA Check

In [ ]:
from config.settings import FRESHNESS_SLA_HOURS

store_with_sla = check_freshness_sla(feature_store)
sla_met = store_with_sla["freshness_sla_met"].mean()
print(f"Freshness SLA ({FRESHNESS_SLA_HOURS}h): {sla_met:.1%} of records meet SLA")

if sla_met < 1.0:
    stale = store_with_sla[~store_with_sla["freshness_sla_met"]]
    print(f"\nStale records: {len(stale)}")
    print(stale[["supplier_id", "year_month", "hours_since_update"]].head())

## 6. Lineage Report

In [ ]:
lineage = get_lineage_report(feature_store)
print(f"Feature store lineage summary:")
print(f"  Rows with ERP features: {lineage['has_erp_features'].sum()}")
print(f"  Rows with LLM features: {lineage['has_llm_features'].sum()}")
print(f"  Coverage: {lineage['has_llm_features'].mean():.1%}")
lineage.head(10)

## 7. Save Feature Store

In [ ]:
feature_store.to_csv("../data/feature_store.csv", index=False)
print("Feature store saved to ../data/feature_store.csv")